## Imports

In [ ]:
import json
import math
import os
import re
import unicodedata
import random
from operator import itemgetter
from pathlib import Path
from pprint import pprint
import pandas as pd
#!pip install --upgrade spacy
import spacy
from spacy.util import compounding, minibatch
from spacy import displacy
# Uncomment if you want Spacy to use GPU for training. Note - this will use transformer architecture
spacy.prefer_gpu()

True

In [ ]:
# import requirements for converting the dataframe to Spacy Docs
from collections import defaultdict
from typing import List
from spacy.language import Language
from spacy.tokens import Doc, DocBin, Span
from spacy.util import filter_spans

## Evaluate Lemmas and POS

In [ ]:
#Lemma evaluator with pos and morph#
 
import pandas as pd
from tqdm import tqdm
import unicodedata as ud
import warnings
import re
from sklearn.metrics import precision_score, recall_score, f1_score
from typing import Dict, List, Tuple, Optional, Any

# Define apostrophes and correct_apostrophe as class variables
apostrophes = ["᾽", "᾿", "'", "'", "'"]
correct_apostrophe = "ʼ"

class LemmaEvaluator:
    @staticmethod
    def clean_and_remove_accents(text: str) -> str:
        """
        Cleans the given text by removing diacritics (accents), except for specific characters,
        and converting it to lowercase.
        """
        allowed_characters = [' ̓', "᾿", "᾽", "'", "'", "'", 'ʼ', '̓']  # Including the Greek apostrophe
        if not isinstance(text, str):
            raise ValueError("Input must be a string.")
        try:
            non_accent_chars = [c for c in ud.normalize('NFKD', text)
            if ud.category(c) != 'Mn' or c in allowed_characters]
            return ''.join(non_accent_chars)
        
        except Exception as e:
            # A more generic exception handling if unexpected errors occur
            print(f"An error occurred: {e}")
            return text
    
    @staticmethod
    def normalize_text(text: str, form: str = 'NFKD',
                      remove_accents: bool = False,
                      lowercase: bool = False,
                      standardize_apostrophe: bool = True,
                      remove_brackets: bool = False,
                      remove_trailing_numbers: bool = False,
                      remove_extra_spaces: bool = False,
                      debug: bool = False) -> str:
        """
        Applies multiple text normalization and cleaning steps on the input text.

        Parameters:
        - text (str): The text to be normalized.
        - form (str): Unicode normalization form ('NFC', 'NFD', 'NFKC', 'NFKD').
        - lowercase (bool): If True, the text is converted to lowercase.
        - standardize_apostrophe (bool): If True, replaces all defined apostrophe characters with a standard one.
        - remove_brackets_only (bool): If True, removes the brackets themselves.
        - remove_trailing_numbers (bool): If True, strips leading or trailing digits from the text.
        
        Returns:
        - str: The processed text.
        """
        normalized_text = text  # Initialize normalized_text with the original text

        # Function to print before and after states for each operation during debugging
        def debug_print(operation_name, before, after):
            if debug:
                print(f"{operation_name} - Before: {before}")
                print(f"{operation_name} - After: {after}")

        # Standardize apostrophe characters if required
        if standardize_apostrophe:
            before_text = normalized_text
            for apos in apostrophes:
                normalized_text = normalized_text.replace(apos, correct_apostrophe)
            debug_print("Standardizing apostrophes", before_text, normalized_text)
            
        if remove_accents:
            before_text = normalized_text
            try:
                normalized_text = LemmaEvaluator.clean_and_remove_accents(normalized_text)
            except Exception as e:
                print(f"An error occurred while removing accents: {e}")
                # Decide what to do here: return the original text, a special value, or stop the process
                return text
            debug_print("Removing accents", before_text, normalized_text)
            
        # Convert to lowercase if required
        if lowercase:
            before_text = normalized_text
            normalized_text = normalized_text.lower()
            debug_print("Lowercase conversion", before_text, normalized_text)

        # Unicode normalization
        if form:
            before_text = normalized_text
            # Handle the form parameter correctly for type checking
            if form == 'NFC':
                normalized_text = ud.normalize('NFC', normalized_text)
            elif form == 'NFD':
                normalized_text = ud.normalize('NFD', normalized_text)
            elif form == 'NFKC':
                normalized_text = ud.normalize('NFKC', normalized_text)
            elif form == 'NFKD':
                normalized_text = ud.normalize('NFKD', normalized_text)
            else:
                # Default to NFKD if somehow an invalid value got through
                normalized_text = ud.normalize('NFKD', normalized_text)
            debug_print("Unicode normalization", before_text, normalized_text)
                
        # Remove brackets only if required
        if remove_brackets:
            before_text = normalized_text
            normalized_text = re.sub(r'[\(\)\[\]]', '', normalized_text)
            debug_print("Removing brackets", before_text, normalized_text)
            
        # Remove trailing numbers if required
        if remove_trailing_numbers:
            before_text = normalized_text
            normalized_text = re.sub(r'^\d+|\d+$', '', normalized_text)
            debug_print("Removing trailing numbers", before_text, normalized_text)

        # Remove multiple spaces and leading/trailing spaces
        if remove_extra_spaces:
            before_text = normalized_text
            normalized_text = ' '.join(normalized_text.split()).strip()
            debug_print("Removing extra spaces", before_text, normalized_text)

        return normalized_text
        
    def __init__(self, models, model_names=None, norm_method='NFKD'):
        self.models = models
        self.model_names = model_names or [f'Model {i+1}' for i in range(len(models))]
        self.norm_method = norm_method

        # Check if normalization method is specified and valid
        if self.norm_method not in ('NFC', 'NFD', 'NFKC', 'NFKD'):
            warnings.warn(f"Invalid normalization method: {self.norm_method}. Using NFKD as default.", UserWarning)
            self.norm_method = 'NFKD'
            
    def normalize_document_text(self, text):
        """
        Normalize the document text using the specified normalization method.
        This ensures consistent encoding throughout the evaluation process.
        """
        # Use the normalize_text method with the specified normalization form
        return self.normalize_text(text, form=self.norm_method, standardize_apostrophe=True)

    def find_token_mapping(self, gold_doc, pred_doc, use_normalization=True):
        """
        Create a mapping between gold and predicted tokens to handle tokenization differences.
        Returns a dictionary mapping gold token indices to predicted token indices.
        
        This enhanced version uses character spans to create more accurate mappings,
        which is especially important for Greek text with different tokenization schemes.
        """
        gold_to_pred = {}
        
        # Get the original text
        original_text = gold_doc.text
        
        # Create character-to-token mappings for both documents
        gold_char_to_token = {}
        for token_idx, token in enumerate(gold_doc):
            for char_idx in range(token.idx, token.idx + len(token.text)):
                gold_char_to_token[char_idx] = token_idx
                
        pred_char_to_token = {}
        for token_idx, token in enumerate(pred_doc):
            for char_idx in range(token.idx, token.idx + len(token.text)):
                pred_char_to_token[char_idx] = token_idx
        
        # Map gold tokens to predicted tokens based on character overlap and normalization
        for gold_idx, gold_token in enumerate(gold_doc):
            # Get character span for this gold token
            start_char = gold_token.idx
            end_char = start_char + len(gold_token.text)
            
            # Find which predicted tokens overlap with this character span
            pred_token_counts = {}
            for char_idx in range(start_char, end_char):
                if char_idx in pred_char_to_token:
                    pred_idx = pred_char_to_token[char_idx]
                    pred_token_counts[pred_idx] = pred_token_counts.get(pred_idx, 0) + 1
            
            # If we found overlapping tokens by character position
            if pred_token_counts:
                best_pred_idx = max(pred_token_counts.items(), key=lambda x: x[1])[0]
                gold_to_pred[gold_idx] = best_pred_idx
                
                # Debug output for significant mismatches
                gold_text = gold_token.text
                pred_text = pred_doc[best_pred_idx].text if best_pred_idx < len(pred_doc) else "N/A"
                if gold_text != pred_text:
                    print(f"Mapped different tokens: Gold[{gold_idx}]='{gold_text}' → Pred[{best_pred_idx}]='{pred_text}'")
            
            # If no overlap found and normalization is enabled, try matching based on normalized text
            elif use_normalization and gold_idx not in gold_to_pred:
                # Normalize the gold token text
                gold_norm = self.normalize_text(gold_token.text, form='NFC', standardize_apostrophe=True, remove_accents=True)
                
                # Try to find a match among predicted tokens
                for pred_idx, pred_token in enumerate(pred_doc):
                    # Skip tokens that are already mapped
                    if pred_idx in gold_to_pred.values():
                        continue
                        
                    # Normalize the predicted token text
                    pred_norm = self.normalize_text(pred_token.text, form='NFC', standardize_apostrophe=True, remove_accents=True)
                    
                    # Check if normalized texts match
                    if gold_norm == pred_norm:
                        gold_to_pred[gold_idx] = pred_idx
                        print(f"Mapped via normalization: Gold[{gold_idx}]='{gold_token.text}' → Pred[{pred_idx}]='{pred_token.text}'")
                        break
        
        # Special handling for apostrophes and other common Greek text issues
        for gold_idx in range(len(gold_doc) - 1):
            if gold_idx not in gold_to_pred and gold_idx + 1 in gold_to_pred:
                # Check if this might be an apostrophe case
                if len(gold_doc[gold_idx].text) == 1 and gold_doc[gold_idx].text in ["'", "ʼ", "'", "᾿", "᾽"]:
                    # Map the apostrophe to the same token as the next token
                    gold_to_pred[gold_idx] = gold_to_pred[gold_idx + 1]
                    print(f"Mapped apostrophe: Gold[{gold_idx}]='{gold_doc[gold_idx].text}' → same as Gold[{gold_idx+1}]")
                
                # Try combining with next token and check if normalized version matches
                elif gold_idx + 1 in gold_to_pred and use_normalization:
                    combined_text = gold_doc[gold_idx].text + gold_doc[gold_idx + 1].text
                    combined_norm = self.normalize_text(combined_text, form='NFC', standardize_apostrophe=True, remove_accents=True)
                    
                    pred_idx = gold_to_pred[gold_idx + 1]
                    if pred_idx < len(pred_doc):
                        pred_norm = self.normalize_text(pred_doc[pred_idx].text, form='NFC', standardize_apostrophe=True, remove_accents=True)
                        
                        if combined_norm == pred_norm:
                            gold_to_pred[gold_idx] = pred_idx
                            print(f"Mapped combined tokens: Gold[{gold_idx}]='{gold_doc[gold_idx].text}' + "
                                  f"Gold[{gold_idx+1}]='{gold_doc[gold_idx+1].text}' → "
                                  f"Pred[{pred_idx}]='{pred_doc[pred_idx].text}'")
        
        return gold_to_pred
        
    def evaluate_lemmas(self, docs, evaluate_lemma=True, evaluate_pos=False, evaluate_tag=False, evaluate_morph=False, use_normalization=True):
        data = []
        skipped_tokens = {
            'total_tokens': 0,
            'empty_gold_lemma': 0,
            'empty_gold_pos': 0,
            'empty_gold_tag': 0,
            'empty_gold_morph': 0,
            'token_length_mismatch': 0
        }

        for doc in tqdm(docs, desc="Processing documents", total=len(docs)):
            # Get the original document text
            doc_text = doc.text
            
            # Normalize the document text before processing
            normalized_text = self.normalize_text(doc_text, form=self.norm_method, standardize_apostrophe=True)
            
            # Process the normalized text with each model
            predicted_docs = [model(normalized_text) for model in self.models]
            
            # Debug: Check tokenization differences
            print("\nTokenization comparison for document:")
            print(f"Original text (first 50 chars): {doc_text[:50]}...")
            print(f"Normalized text (first 50 chars): {normalized_text[:50]}...")
            print(f"Gold doc tokens: {[token.text for token in doc][:10]}...")
            
            # Create token mappings for each model
            token_mappings = []
            for i, (model_name, predicted_doc) in enumerate(zip(self.model_names, predicted_docs)):
                print(f"{model_name} tokens: {[token.text for token in predicted_doc][:10]}...")
                
                # Create mapping between gold and predicted tokens
                mapping = self.find_token_mapping(doc, predicted_doc, use_normalization=use_normalization)
                token_mappings.append(mapping)
                
                # Print some mapping examples for debugging
                print(f"Token mapping examples for {model_name}:")
                for gold_idx, pred_idx in list(mapping.items())[:5]:
                    if gold_idx < len(doc) and pred_idx < len(predicted_doc):
                        print(f"  Gold[{gold_idx}]: '{doc[gold_idx].text}' (lemma: '{doc[gold_idx].lemma_}') → "
                              f"Pred[{pred_idx}]: '{predicted_doc[pred_idx].text}' (lemma: '{predicted_doc[pred_idx].lemma_}')")
                
                # Check for unmapped tokens
                unmapped = [idx for idx in range(len(doc)) if idx not in mapping]
                if unmapped:
                    print(f"  Warning: {len(unmapped)} unmapped gold tokens for {model_name}")
                    for idx in unmapped[:3]:  # Show first few unmapped tokens
                        print(f"    Unmapped Gold[{idx}]: '{doc[idx].text}' (lemma: '{doc[idx].lemma_}')")
            
            for token_idx, token in enumerate(doc):
                skipped_tokens['total_tokens'] += 1
                
                # Skip tokens with empty or invalid data based on evaluation flags
                skip_token = False
                
                if evaluate_lemma and (not token.lemma_ or token.lemma_.strip() == ''):
                    skipped_tokens['empty_gold_lemma'] += 1
                    skip_token = True
                
                if evaluate_pos and (not token.pos_ or token.pos_.strip() == ''):
                    skipped_tokens['empty_gold_pos'] += 1
                    skip_token = True
                
                if evaluate_tag and (not token.tag_ or token.tag_.strip() == ''):
                    skipped_tokens['empty_gold_tag'] += 1
                    skip_token = True
                
                if evaluate_morph:
                    morph_dict = token.morph.to_dict()
                    if not morph_dict or all(not value for value in morph_dict.values()):
                        skipped_tokens['empty_gold_morph'] += 1
                        skip_token = True
                
                if skip_token:
                    continue

                token_data = {"Text": doc.text, "Token": token.text}

                if evaluate_lemma:
                    token_data["Gold Lemma"] = token.lemma_
                    for i, (model_name, predicted_doc) in enumerate(zip(self.model_names, predicted_docs)):
                        # Get the mapped token index from our mapping
                        mapped_idx = token_mappings[i].get(token_idx)
                        
                        if mapped_idx is not None and mapped_idx < len(predicted_doc):
                            # Use the mapped token
                            token_data[f"{model_name} Lemma"] = predicted_doc[mapped_idx].lemma_
                        elif token_idx < len(predicted_doc):
                            # Fall back to index-based alignment
                            token_data[f"{model_name} Lemma"] = predicted_doc[token_idx].lemma_
                        else:
                            token_data[f"{model_name} Lemma"] = "N/A"
                            skipped_tokens['token_length_mismatch'] += 1

                if evaluate_pos:
                    token_data["Gold POS"] = token.pos_
                    for i, (model_name, predicted_doc) in enumerate(zip(self.model_names, predicted_docs)):
                        # Get the mapped token index from our mapping
                        mapped_idx = token_mappings[i].get(token_idx)
                        
                        if mapped_idx is not None and mapped_idx < len(predicted_doc):
                            # Use the mapped token
                            token_data[f"{model_name} POS"] = predicted_doc[mapped_idx].pos_
                        elif token_idx < len(predicted_doc):
                            # Fall back to index-based alignment
                            token_data[f"{model_name} POS"] = predicted_doc[token_idx].pos_
                        else:
                            token_data[f"{model_name} POS"] = "N/A"
                            skipped_tokens['token_length_mismatch'] += 1

                if evaluate_tag:
                    token_data["Gold TAG"] = token.tag_
                    for i, (model_name, predicted_doc) in enumerate(zip(self.model_names, predicted_docs)):
                        # Get the mapped token index from our mapping
                        mapped_idx = token_mappings[i].get(token_idx)
                        
                        if mapped_idx is not None and mapped_idx < len(predicted_doc):
                            # Use the mapped token
                            token_data[f"{model_name} TAG"] = predicted_doc[mapped_idx].tag_
                        elif token_idx < len(predicted_doc):
                            # Fall back to index-based alignment
                            token_data[f"{model_name} TAG"] = predicted_doc[token_idx].tag_
                        else:
                            token_data[f"{model_name} TAG"] = "N/A"
                            skipped_tokens['token_length_mismatch'] += 1

                if evaluate_morph:
                    token_data["Gold Morph"] = str(token.morph.to_dict())
                    for i, (model_name, predicted_doc) in enumerate(zip(self.model_names, predicted_docs)):
                        # Get the mapped token index from our mapping
                        mapped_idx = token_mappings[i].get(token_idx)
                        
                        if mapped_idx is not None and mapped_idx < len(predicted_doc):
                            # Use the mapped token
                            token_data[f"{model_name} Morph"] = str(predicted_doc[mapped_idx].morph.to_dict())
                        elif token_idx < len(predicted_doc):
                            # Fall back to index-based alignment
                            token_data[f"{model_name} Morph"] = str(predicted_doc[token_idx].morph.to_dict())
                        else:
                            token_data[f"{model_name} Morph"] = "N/A"
                            skipped_tokens['token_length_mismatch'] += 1

                data.append(token_data)

        # Detailed logging about skipped tokens
        print("\nToken Skipping Statistics:")
        print(f"Total tokens processed: {skipped_tokens['total_tokens']}")
        
        if evaluate_lemma:
            print(f"Tokens skipped due to empty gold lemma: {skipped_tokens['empty_gold_lemma']} ({skipped_tokens['empty_gold_lemma']/skipped_tokens['total_tokens']*100:.2f}%)")
        
        if evaluate_pos:
            print(f"Tokens skipped due to empty gold POS: {skipped_tokens['empty_gold_pos']} ({skipped_tokens['empty_gold_pos']/skipped_tokens['total_tokens']*100:.2f}%)")
        
        if evaluate_tag:
            print(f"Tokens skipped due to empty gold TAG: {skipped_tokens['empty_gold_tag']} ({skipped_tokens['empty_gold_tag']/skipped_tokens['total_tokens']*100:.2f}%)")
        
        if evaluate_morph:
            print(f"Tokens skipped due to empty gold Morphology: {skipped_tokens['empty_gold_morph']} ({skipped_tokens['empty_gold_morph']/skipped_tokens['total_tokens']*100:.2f}%)")
        
        print(f"Tokens skipped due to token length mismatch: {skipped_tokens['token_length_mismatch']} ({skipped_tokens['token_length_mismatch']/skipped_tokens['total_tokens']*100:.2f}%)")

        # Create a DataFrame from the token-level data
        columns = ["Text", "Token"]
        if evaluate_lemma:
            columns += ["Gold Lemma"] + [f"{name} Lemma" for name in self.model_names]
        if evaluate_pos:
            columns += ["Gold POS"] + [f"{name} POS" for name in self.model_names]
        if evaluate_tag:
            columns += ["Gold TAG"] + [f"{name} TAG" for name in self.model_names]
        if evaluate_morph:
            columns += ["Gold Morph"] + [f"{name} Morph" for name in self.model_names]
        
        # Ensure that the number of columns matches the data
        assert len(columns) == len(data[0]), f"{len(columns)} columns specified, but data has {len(data[0])} columns"

        df = pd.DataFrame(data, columns=columns)
        
        # Calculate precision, recall, and F1-score for each model
        metrics = []
        for i in range(len(self.models)):
            model_metrics = [self.model_names[i]]
            print("modl metrics", model_metrics)
            if evaluate_lemma:
                gold_lemmas = df["Gold Lemma"].tolist()
                predicted_lemmas = df[f"{self.model_names[i]} Lemma"].tolist()
                lemma_precision = precision_score(gold_lemmas, predicted_lemmas, average='weighted', zero_division=0)
                lemma_recall = recall_score(gold_lemmas, predicted_lemmas, average='weighted', zero_division=0)
                lemma_f1 = f1_score(gold_lemmas, predicted_lemmas, average='weighted', zero_division=0)
                model_metrics.extend([lemma_precision, lemma_recall, lemma_f1])

            if evaluate_pos:
                gold_pos = df["Gold POS"].tolist()
                predicted_pos = df[f"{self.model_names[i]} POS"].tolist()
                pos_precision = precision_score(gold_pos, predicted_pos, average='weighted', zero_division=0)
                pos_recall = recall_score(gold_pos, predicted_pos, average='weighted', zero_division=0)
                pos_f1 = f1_score(gold_pos, predicted_pos, average='weighted', zero_division=0)
                model_metrics.extend([pos_precision, pos_recall, pos_f1])

            if evaluate_tag:
                gold_tags = df["Gold TAG"].tolist()
                predicted_tags = df[f"{self.model_names[i]} TAG"].tolist()
                tag_precision = precision_score(gold_tags, predicted_tags, average='weighted', zero_division=0)
                tag_recall = recall_score(gold_tags, predicted_tags, average='weighted', zero_division=0)
                tag_f1 = f1_score(gold_tags, predicted_tags, average='weighted', zero_division=0)
                model_metrics.extend([tag_precision, tag_recall, tag_f1])

            if evaluate_morph:
                # Implement evaluation metrics for morphological features if needed
                gold_morph = df["Gold Morph"].tolist()
                predicted_morph = df[f"{self.model_names[i]} Morph"].tolist()
                morph_precision = precision_score(gold_morph, predicted_morph, average='weighted', zero_division=0)
                morph_recall = recall_score(gold_morph, predicted_morph, average='weighted', zero_division=0)
                morph_f1 = f1_score(gold_morph, predicted_morph, average='weighted', zero_division=0)
                model_metrics.extend([morph_precision, morph_recall, morph_f1])
                
            metrics.append(model_metrics)

        # Print out the evaluation metrics in a table format
        columns_metrics = ["Model", "Lemma Precision", "Lemma Recall", "Lemma F1-Score"]
        if evaluate_pos:
            columns_metrics.extend(["POS Precision", "POS Recall", "POS F1-Score"])
        if evaluate_tag:
            columns_metrics.extend(["TAG Precision", "TAG Recall", "TAG F1-Score"])
        if evaluate_morph:
            columns_metrics.extend(["Morph Precision", "Morph Recall", "Morph F1-Score"])
        df_metrics = pd.DataFrame(metrics, columns=columns_metrics)

        print(df_metrics.to_string(index=False))

        return df

The evaluation function evaluates between models defined in the following cell.

In [ ]:
nlp1 = spacy.load('../training/model1_folder')
nlp2 = spacy.load('../training/model2_folder')

# give a name to the models  
nlp1.meta['name'] = 'name of model 1'
nlp2.meta['name'] = 'name of model 2'

In [ ]:
#load a base model
nlp_grecy = spacy.load("grc_proiel_trf")
models = [nlp1, nlp2]
model_names = [model.meta.get('name', f'Model {i+1}') for i, model in enumerate(models)]

evaluator = LemmaEvaluator(models, model_names, norm_method="NFC")

# Load the test data
test_docs = DocBin().from_disk('../corpus/test/lemma_test/test_lemma_NFC.spacy')
docs = list(test_docs.get_docs(nlp_grecy.vocab))


Run the evaluations.
Choose which evaluations to run: lemma, pos, tag or morph.
Choose if you want to normalize data as well.

In [ ]:
# POS and Lemma Evaluations
df_evaluate_lemma = evaluator.evaluate_lemmas(docs, evaluate_lemma=True, evaluate_pos=True, evaluate_tag=False, evaluate_morph=False, use_normalization=True)

## Evaluate Span Categorizer

In [ ]:
#spancat evaluator#

import warnings
import unicodedata as ud
import re
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score

class SPANEvaluator:
    def __init__(self, models, norm_method=None):
        self.models = models
        self.norm_method = norm_method
        
        # check if normalization method is specified
        if self.norm_method is None:
            warnings.warn("Normalization method not specified. Text may not be normalized correctly.", UserWarning)

    def evaluate_SpanCat(self, docs):

        data = []
        same_span = 0
        diff_span = 0
        correct_spans = [[] for _ in range(len(self.models))]
        gold_labels = []
        predicted_labels = [[] for _ in range(len(self.models))]


        for doc in tqdm(docs, desc="Evaluating models", total=len(docs)):
            doc_text = doc.text
            docs = [model(doc_text) for model in self.models]

            # Get the gold spans, their start, end, and label
            gold_spans = []
            for span in doc.spans["sc"]:
                gold_spans.append({
                    "start": span.start_char,
                    "end": span.end_char,
                    "label": span.label_,
                    "token": doc.text[span.start_char:span.end_char]
                })

            for gold_span in gold_spans:
                gold_label = gold_span["label"]
                token = gold_span["token"]
                predicted_labels_for_span = []
                
                span_group = docs[0].spans["sc"]
                print(f"SpanGroup '{span_group.name}' contains {len(span_group)} spans:")
                for span in span_group:
                    print(f"- Span: '{span.text}' [{span.start_char}, {span.end_char}], Labels: {span.label_}")

                for i, doc_model in enumerate(docs):
                    predicted_label = None
                    for span in doc_model.spans["sc"]:
                        # print gold span details and predicted span details
                        print(f"Gold span: {token}, {gold_span['start']} - {gold_span['end']}, {gold_label}")
                        print(f"Predicted span: {span.text}, {span.start_char} - {span.end_char}, {span.label_}")

                        # match the predicted span with the gold span
                        if span.start_char == gold_span["start"] and span.end_char == gold_span["end"]:
                            predicted_label = span.label_
                            print(f"Model {i+1} predicted label: {predicted_label}")
                            break
                        elif abs(span.start_char - gold_span["start"]) <= 5 and abs(span.end_char - gold_span["end"]) <= 5:
                            predicted_label = span.label_
                            print(f"Model {i+1} predicted label: None")
                            break
                        #print(f"- Span: '{span.text}' [{span.start_char}, {span.end_char}], Labels: {span.label_}")
                    predicted_labels_for_span.append(predicted_label)

                # Analyze and compare labels
                if len(set(predicted_labels_for_span)) == 1 and predicted_labels_for_span[0] == gold_label:
                    label_comparison = "All span labels are the same, correct label is {}".format(gold_label)
                    same_span += 1
                else:
                    label_comparison = "Span labels are different, correct label is {}".format(gold_label)
                    diff_span += 1
                    for i, predicted_label in enumerate(predicted_labels_for_span):
                        if predicted_label != gold_label:
                            label_comparison += ", Model {} predicted: {}".format(i+1, predicted_label)

                for i, predicted_label in enumerate(predicted_labels_for_span):
                    correct_spans[i].append(predicted_label == gold_label)


                gold_labels.append(gold_label)
                for i, predicted_label in enumerate(predicted_labels_for_span):
                    predicted_labels[i].append(predicted_label)

                data.append({
                    "Text": doc.text,
                    "Token": token,
                    "Gold Label": gold_label,
                    **{f"Model {i+1} Label": label for i, label in enumerate(predicted_labels_for_span)},
                    "Result": label_comparison
                })

        # Create a DataFrame from the data
        columns = ["Text", "Token", "Gold Label"] + [f"Model {i+1} Label" for i in range(len(self.models))] + ["Result"]
        df_evaluate_spans = pd.DataFrame(data, columns=columns)
        predicted_labels = [[label if label is not None else 'None' for label in model_labels] for model_labels in predicted_labels]
        
        # Evaluation metrics
        total = same_span + diff_span
        print(f"Total same NER labels: {same_span} ({same_span/total:.2%})")
        print(f"Total different NER labels: {diff_span} ({diff_span/total:.2%})\n")

        metrics = []
        for i, model in enumerate(self.models):
            precision = precision_score(gold_labels, predicted_labels[i], average='weighted', zero_division=1)
            recall = recall_score(gold_labels, predicted_labels[i], average='weighted', zero_division=1)
            f1 = f1_score(gold_labels, predicted_labels[i], average='weighted', zero_division=1)
            try:
                accuracy = sum(correct_spans[i]) / len(correct_spans[i])
            except ZeroDivisionError:
                accuracy = 0.00
            metrics.append([f"Model {i+1}", precision, recall, f1, accuracy])

        df_metrics = pd.DataFrame(metrics, columns=["Model", "Precision", "Recall", "F1-Score", "Accuracy"])
        print(df_metrics.to_string(index=False))


        return df_evaluate_spans
    

    def clean_text(self, text):
        # Check if the normalization method is valid
        if self.norm_method is not None and self.norm_method not in ['NFD', 'NFC', 'NFKD', 'NFKC']:
            raise ValueError("Normalization method is not valid. Must be one of ['NFD', 'NFC', 'NFKD', 'NFKC'].")
        elif self.norm_method is not None:
            cleaned = re.sub(r"[\(\[].*?[\)\]]", "", text)
            cleaned = ud.normalize(self.norm_method, cleaned)
        else:
            cleaned = text
        return cleaned

The evaluation function evaluates between models defined in the following cell.

In [ ]:
#Models Span Evaluations#

import spacy
from spacy.tokens import Doc, DocBin, Span
spacy.prefer_gpu()

#Span Evaluations:
nlp1 = spacy.load('../training/model1_folder')
nlp2 = spacy.load('../training/model2_folder')

# define threshold for spancat
for nlp in [nlp1, nlp2]:
    nlp.get_pipe("spancat").cfg["threshold"] = 0.25
    print(nlp.get_pipe("spancat").cfg)

#if you are adding a new spancat to the pipeline, uncomment the following lines
#nlp1.add_pipe("spancat", config={"threshold": 0.25}, last=True)
#spancat.cfg["spans_key"] = "sc"

models = [nlp1, nlp2]

evaluator = SPANEvaluator(models, norm_method='NFC')

test_docs = DocBin().from_disk('../corpus/test/spancat_test/spancat_test_NFC.spacy')
docs = list(test_docs.get_docs(nlp1.vocab))

df_evaluate_spans = evaluator.evaluate_SpanCat(docs)

In [ ]:
# Span evaluation for a specific given sentence

import spacy
from spacy.tokens import DocBin
FORMAT = 'NFC'

nlp = spacy.load('../training/model_folder')
nlp.get_pipe("spancat").cfg["threshold"] = 0.2
nlp.get_pipe("spancat").cfg

train_docbin = DocBin().from_disk("../corpus/train/spancat_train/spancat_train_{0}.spacy".format(FORMAT))
# get docs from new_docbin
train_docs = list(train_docbin.get_docs(nlp.vocab))
dev_docbin = DocBin().from_disk("../corpus/dev/spancat_dev/spancat_dev_{0}.spacy".format(FORMAT))
# get docs from new_docbin
dev_docs = list(dev_docbin.get_docs(nlp.vocab))
test_docbin = DocBin().from_disk("../corpus/test/spancat_test/spancat_test_{0}.spacy".format(FORMAT))
# get docs from new_docbin
test_docs = list(test_docbin.get_docs(nlp.vocab))
# count sentences in train, test and dev data
print ("train:", len(train_docs), "dev:", len(dev_docs), "test:", len(test_docs))

# find sentence and print spans
for doc in test_docs:
    
    # checking a sentence that had a word with two labels
    if doc.text == "ἅμα δʼ ἡ ἀνάπνευσις καὶ ἔκπνευσις γίνεται εἰς τὸ στῆθος, καὶ ἀδύνατον χωρὶς τοῖς μυκτῆρσιν ἀναπνεῦσαι ἢ ἐκπνεῦσαι, διὰ τὸ ἐκ τοῦ στήθους εἶναι τὴν ἀναπνοὴν καὶ ἐκπνοὴν κατὰ τὸν γαργαρεῶνα, καὶ μὴ ἐκ τῆς κεφαλῆς τινι μέρει":
        print("doc user data: ", doc.user_data)
        
        # find spans that have the same token and more than one label
        print(doc.text)
        #for span in doc.spans["sc"]:
            #print(span.text, span.start_char, span.end_char, span.label_)
            # print character text in location of indices of span
            #print(doc.text[span.start_char:span.end_char])      
        
        # print spansgroup assuming you have a SpanGroup named 'sc' in doc.spans
        span_group = doc.spans["sc"]
        
        print(f"SpanGroup '{span_group.name}' contains {len(span_group)} spans:")
        for span in span_group:
            print(f"- Span: '{span.text}' [{span.start_char}, {span.end_char}], Labels: {span.label_}")
            
        perdicted = nlp(doc.text)
        
        # print spansgroup assuming you have a SpanGroup named 'sc' in doc.spans
        span_group = perdicted.spans["sc"]
        print(f"SpanGroup '{span_group.name}' contains {len(span_group)} perdicted spans:")
        print(f" Scores: {span_group.attrs['scores']}")

        for i, span in enumerate(span_group):
            score = span_group.attrs["scores"][i]
            print(f"- Span: '{span.text}' [{span.start_char}, {span.end_char}], Labels: {span.label_}, Score: {score:.4f}")



# Visualize inference of a text
The following part creates a visualization of a given sentence. It will show lemmata, dependencies and spans categories for a given sentence.

In [ ]:
import unicodedata as ud
import re
import warnings

def normalize_text(text: str, form: str = 'NFKD',
                  remove_accents: bool = False,
                  lowercase: bool = False,
                  standardize_apostrophe: bool = True,
                  remove_brackets: bool = False,
                  remove_trailing_numbers: bool = False,
                  remove_extra_spaces: bool = False,
                  debug: bool = False) -> str:
    """
    Normalizes Greek text with various options.
    """
    normalized_text = text
    correct_apostrophe = "'"
    apostrophes = ["'", "'", "ʼ", "`", "´", "'"]

    # Debug printing helper
    def debug_print(operation_name, before, after):
        if debug:
            print(f"{operation_name} - Before: {before}")
            print(f"{operation_name} - After: {after}")

    # Standardize apostrophes
    if standardize_apostrophe:
        before_text = normalized_text
        for apos in apostrophes:
            normalized_text = normalized_text.replace(apos, correct_apostrophe)
        debug_print("Standardizing apostrophes", before_text, normalized_text)

    # Unicode normalization
    if form:
        before_text = normalized_text
        try:
            normalized_text = ud.normalize(form, normalized_text)
            debug_print("Unicode normalization", before_text, normalized_text)
        except Exception as e:
            warnings.warn(f"Unicode normalization failed: {e}")
            
    # Apply other normalizations as needed
    if lowercase:
        normalized_text = normalized_text.lower()
    if remove_brackets:
        normalized_text = re.sub(r'[\(\)\[\]]', '', normalized_text)
    if remove_trailing_numbers:
        normalized_text = re.sub(r'^\d+|\d+$', '', normalized_text)
    if remove_extra_spaces:
        normalized_text = ' '.join(normalized_text.split()).strip()

    return normalized_text


In [ ]:
# Normalize the input text before processing
text = "ἡ δ’ ἄνωθεν ἀρχὴ τοῦδε τοῦ μυὸς, ἣν ὀνομάζουσι κεφαλὴν αὐτοῦ τὴν ἔκφυσιν ἔχει σαρκώδη κατὰ μέσην μάλιστα τὴν ῥάχιν τοῦ τῆς λαγόνος ὀστοῦ, μακρὰ δὲ καὶ αὐτὴ κατὰ τὸ μῆκος ἐκτέταται τοῦ ζῴου, προπετὴς ἐπὶ τῶν ἰσχνῶν φαινομένη πᾶσι καὶ πρὸ τῆς ἀνατομῆς. καὶ μέντοι καὶ διορίζουσ΄ αὐτῆς ἀπὸ τῶν ὀπίσω μερῶν τὰ πρόσω τελευτᾷ καθ’ ὅλον τὸ μῆκος εἰς ἄκανθαν ὀξεῖαν, οἵα πέρ ἐστι καὶ ἡ τῆς ὠμοπλάτης ῥάχις ἐν τοῖς ὑψηλοτάτοις ἑαυτῆς."

normalized_text = normalize_text(
    text,
    form='NFC',
    standardize_apostrophe=True,
    remove_extra_spaces=True,
    debug=True  # Set to True to see the normalization steps
)


In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load("../training/model_folder")

In [ ]:
# Input text
doc = nlp(normalized_text)
sentence_spans=list(doc.sents)

In [ ]:
# colors options for span visualization
span_options = {
    "spans_key": "sc",
    "colors": {
        "Body Part": "#ff9999",
        "Adjectives/Qualities": "#99ccff",
        "Topography": "#ffcc99",
        "Medical": "#ccffcc",
        "Pathology": "#ffccff",
        "Physiology": "#ffff99",
        "Technical Appellation": "#99ff99",
        "Division": "#ff9966",
        "Action Verbs": "#66ccff",
        "Symmetry/Opposition": "#ff6666",
        "Technical appellation": "#6699ff"
    }
}

span_html = displacy.render(doc, style="span", options=span_options, jupyter=False)

with open("span_html.html", "w", encoding="utf-8") as f:
    f.write(span_html)
# Dependency visualization options
dep_options = {
    "compact": True,
    "color": "#2b6cb0",
    "bg": "#ffffff",
    "font": "Source Sans Pro",
    "offset_x": 20,
    "arrow_stroke": 3,
    "arrow_width": 10,
    "arrow_spacing": 0,
    "word_spacing": 35,
    "distance": 150,
    "add_lemma": True,
    "collapse_punct": True,
    "fine_grained": False
}


# Render dependency visualization
dep_html = displacy.render(sentence_spans, style="dep", options=dep_options, jupyter=False)

# Extract lemmas
lemma_table = "<table border='1'><tr><th>Token</th><th>Lemma</th><th>POS</th></tr>"
for token in doc:
    lemma_table += f"<tr><td>{token.text}</td><td>{token.lemma_}</td><td>{token.pos_}</td></tr>"
lemma_table += "</table>"

# Combined HTML template with improved styling
combined_html = f"""
<html>
<head>
    <title>Combined Linguistic Analysis</title>
    <link href="https://fonts.googleapis.com/css2?family=Source+Sans+Pro:wght@400;600&display=swap" rel="stylesheet">
    <style>
        body {{
            font-family: 'Source Sans Pro', sans-serif;
            line-height: 1.8;
            margin: 40px;
            max-width: 1200px;
            padding: 0 20px;
            background-color: #f8fafc;
            color: #1a202c;
        }}
        
        h1 {{
            color: #2d3748;
            border-bottom: 2px solid #e2e8f0;
            padding-bottom: 10px;
            margin-top: 0px;
            font-size: 2em;
        }}
        
        .section {{
            background: white;
            padding: 25px;
            border-radius: 8px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.12);
            margin: 1px 0;
        }}
        
        /* New styles for dependency visualization */
        .section svg.displacy {{
            transform: scale(0.9);  /* Slightly reduce overall size */
            transform-origin: top left;
            margin: 0 !important;
            height: auto !important;  /* Let height be determined by content */
        }}
        
        .section svg.displacy text {{
            font-size: 14px !important;
        }}
        
        /* Adjust spacing between sections */
        .section + .section {{
        margin-top: 20px;
        }}


        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 25px 0;
            font-size: 1.1em;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 3px rgba(0,0,0,0.12);
        }}
        
        th, td {{
            padding: 12px 15px;
            text-align: left;
        }}
        
        th {{
            background-color: #4299e1;
            color: white;
            font-weight: 600;
        }}
        
        tr:nth-child(even) {{
            background-color: #f7fafc;
        }}
        
        tr:hover {{
            background-color: #ebf4ff;
        }}
    </style>
</head>
<body>
    <h1>Spans Analysis</h1>
    <div class="section">
        {span_html}
    </div>
    
    <h1>Dependency Analysis</h1>
    <div class="section dep-section">
        {dep_html}
    </div>
    
    <h1>Lemma Analysis</h1>
    <div class="section">
        {lemma_table}
    </div>
</body>
</html>
"""

# Save the visualization
with open("span_analysis.html", "w", encoding="utf-8") as f:
    f.write(combined_html)
    
print("Combined visualization saved to 'span_analysis.html'. Open it in a browser to view.")

## Lemma and depndency parsing visualization
This part only visualizes the lemma and dependencies.

In [ ]:
import spacy
from spacy import displacy


nlp = spacy.load("../training/model_folder")

# Input text
text = "ἡ δ’ ἄνωθεν ἀρχὴ τοῦδε τοῦ μυὸς, ἣν ὀνομάζουσι κεφαλὴν αὐτοῦ τὴν ἔκφυσιν ἔχει σαρκώδη κατὰ μέσην μάλιστα τὴν ῥάχιν τοῦ τῆς λαγόνος ὀστοῦ, μακρὰ δὲ καὶ αὐτὴ κατὰ τὸ μῆκος ἐκτέταται τοῦ ζῴου, προπετὴς ἐπὶ τῶν ἰσχνῶν φαινομένη πᾶσι καὶ πρὸ τῆς ἀνατομῆς. καὶ μέντοι καὶ διορίζουσ΄ αὐτῆς ἀπὸ τῶν ὀπίσω μερῶν τὰ πρόσω τελευτᾷ καθ’ ὅλον τὸ μῆκος εἰς ἄκανθαν ὀξεῖαν, οἵα πέρ ἐστι καὶ ἡ τῆς ὠμοπλάτης ῥάχις ἐν τοῖς ὑψηλοτάτοις ἑαυτῆς."
doc = nlp(text)
sentence_spans=list(doc.sents)

# Dependency visualization options
dep_options = {
    "compact": True,
    "color": "#2b6cb0",
    "bg": "#ffffff",
    "font": "Source Sans Pro",
    "offset_x": 75,
    "arrow_stroke": 3,
    "arrow_width": 10,
    "arrow_spacing": 15,
    "word_spacing": 50,
    "distance": 175,
    "add_lemma": True,
    "collapse_punct": True,
    "fine_grained": False
}


# Render dependency visualization
dep_html = displacy.render(sentence_spans, style="dep", options=dep_options, jupyter=False)

# Extract lemmas
lemma_table = "<table border='1'><tr><th>Token</th><th>Lemma</th><th>POS</th></tr>"
for token in doc:
    lemma_table += f"<tr><td>{token.text}</td><td>{token.lemma_}</td><td>{token.pos_}</td></tr>"
lemma_table += "</table>"

# Combined HTML template with improved styling
combined_html = f"""
<html>
<head>
    <title>Combined Linguistic Analysis</title>
    <link href="https://fonts.googleapis.com/css2?family=Source+Sans+Pro:wght@400;600&display=swap" rel="stylesheet">
    <style>
        body {{
            font-family: 'Source Sans Pro', sans-serif;
            line-height: 1.8;
            margin: 40px auto;
            max-width: 1200px;
            padding: 0 20px;
            background-color: #f8fafc;
            color: #1a202c;
        }}
        
        h1 {{
            color: #2d3748;
            border-bottom: 2px solid #e2e8f0;
            padding-bottom: 10px;
            margin-top: 40px;
            font-size: 2em;
        }}
        
        .section {{
            background: white;
            padding: 25px;
            border-radius: 8px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.12);
            margin: 20px 0;
        }}
        
        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 25px 0;
            font-size: 1.1em;
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 1px 3px rgba(0,0,0,0.12);
        }}
        
        th, td {{
            padding: 12px 15px;
            text-align: left;
        }}
        
        th {{
            background-color: #4299e1;
            color: white;
            font-weight: 600;
        }}
        
        tr:nth-child(even) {{
            background-color: #f7fafc;
        }}
        
        tr:hover {{
            background-color: #ebf4ff;
        }}
    </style>
</head>
<body>
    <h1>Dependency Analysis</h1>
    <div class="section">
        {dep_html}
    </div>
    
    <h1>Lemma Analysis</h1>
    <div class="section">
        {lemma_table}
    </div>
</body>
</html>
"""

# Save the visualization
with open("dep_analysis.html", "w", encoding="utf-8") as f:
    f.write(combined_html)
    
print("Combined visualization saved to 'dep_analysis.html'. Open it in a browser to view.")